In [1]:
from unsloth import FastLanguageModel

def load_model(model_name: str, max_sequence_length: int = 2048, load_in_4bit: bool = True):

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = model_name,
        max_seq_length = max_sequence_length,
        load_in_4bit = load_in_4bit,  #
        # token = "YOUR_HF_TOKEN", # HF Token for gated models
    )
    return model, tokenizer


model, tokenizer = load_model("unsloth/mistral-7b-instruct-v0.3-bnb-4bit")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/anaconda/envs/finetune_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.2.1: Fast Mistral patching. Transformers: 4.57.6.
   \\   /|    Tesla V100-PCIE-16GB. Num GPUs = 1. Max memory: 15.773 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [5]:
# === Verify: no double BOS in training or inference prompts ===
import sys; sys.path.insert(0, ".")
from finetuning.prompts import build_training_text, build_inference_prompt

# 1) Check inference prompt text — should NOT start with <s>
inf_text = build_inference_prompt(tokenizer, "Hallo wereld")
print("Inference prompt starts with <s>?", inf_text.startswith(tokenizer.bos_token))
print("Inference prompt (first 80 chars):", repr(inf_text[:80]))

# 2) Tokenize the way inference.py does (add_special_tokens=True, the default)
inf_tokens = tokenizer(inf_text, add_special_tokens=True)
first_tokens = tokenizer.convert_ids_to_tokens(inf_tokens["input_ids"][:8])
print("Inference token IDs (first 8):", inf_tokens["input_ids"][:8])
print("Inference tokens   (first 8):", first_tokens)
assert first_tokens.count("<s>") == 1, f"DOUBLE BOS in inference! {first_tokens}"

# 3) Check training text — same BOS guarantee
train_text = build_training_text(tokenizer, "Hallo wereld", "Dit is het antwoord.")
print("\nTraining text starts with <s>?", train_text.startswith(tokenizer.bos_token))
train_tokens = tokenizer(train_text, add_special_tokens=True)
first_train = tokenizer.convert_ids_to_tokens(train_tokens["input_ids"][:8])
print("Training token IDs (first 8):", train_tokens["input_ids"][:8])
print("Training tokens   (first 8):", first_train)
assert first_train.count("<s>") == 1, f"DOUBLE BOS in training! {first_train}"

print("\n✅ Single BOS confirmed for both training and inference")

Inference prompt starts with <s>? False
Inference prompt (first 80 chars): '[INST] Beantwoord de volgende vraag zo goed mogelijk in het Nederlands.\n\nHallo w'
Inference token IDs (first 8): [1, 3, 2507, 1208, 1577, 1324, 1108, 2976]
Inference tokens   (first 8): ['<s>', '[INST]', '▁Be', 'ant', 'wo', 'ord', '▁de', '▁vol']

Training text starts with <s>? False
Training token IDs (first 8): [1, 3, 2507, 1208, 1577, 1324, 1108, 2976]
Training tokens   (first 8): ['<s>', '[INST]', '▁Be', 'ant', 'wo', 'ord', '▁de', '▁vol']

✅ Single BOS confirmed for both training and inference


In [ ]:
from finetuning.prompts import _strip_bos

SYSTEM_PROMPT = "Je bent een behulpzame assistent. Antwoord altijd in het Nederlands. Wees beknopt en relevant."
user_question = "Geef de verschillen aan tussen een tropisch en gematigd regenwoud"

messages = [
    {"role": "user", "content": f"{SYSTEM_PROMPT}\n\n{user_question}"},
]

# tokenize=False + strip BOS + tokenizer() — matches inference.py exactly
prompt_str = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
prompt_str = _strip_bos(prompt_str, tokenizer)
inputs = tokenizer(prompt_str, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
    repetition_penalty=1.15,
    use_cache=True,
)

# decode only generated tokens
input_len = inputs["input_ids"].shape[-1]
print(tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip())

Tropische en gematigde regenwouden verschillen voornamelijk door hun locatie, klimaat en flora en fauna.

1. Locatie: Tropische regenwouden zijn gevestigd rond de evenaar (bijvoorbeeld Amazone-regenwoud) en hebben een warm en nat klimaat met jaarlijks meer dan 2000 mm neerslag. Gematigde regenwouden liggen op hogere breedtegraden (bijvoorbeeld West-Canada) en hebben een gematigd klimaat met minder neerslag vergeleken met tropische regenwouden.

2. Flora en Fauna: De biodiversiteit is hoog in zowel tropische als gematigde regenwouden, maar er zijn wel verschillen. In tropische regenwouden vindt men veel exotische planten en dieren die niet elders ter wereld voorkomen. In gematigde regenwouden zijn de soorten meestal minder exotisch, maar er zijn toch nog veel endemische soorten te vinden.

3. Structuren: De structuren van de bossen verschillen ook. Tropische regenwouden hebben vaak een horizontale structuur met een dichte ondergroei en een lage canopy. Gematigde regenwouden hebben vaak 

In [13]:

user_question = "Explain AI in 2 phrases"

messages = [
    {"role": "user", "content": user_question},
]
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")



outputs = model.generate(
    inputs,
    max_new_tokens=600,                  # hard stop
    eos_token_id=tokenizer.eos_token_id, # stop on EOS
    #pad_token_id=tokenizer.pad_token_id,
    #use_cache=True,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=False))

<s>[INST] Explain AI in 2 phrases[/INST] 1. AI is a branch of computer science that enables machines to mimic intelligent human behavior.

2. AI uses algorithms and statistical models to analyze data, make decisions, and solve complex problems.</s>


In [ ]:
from openai import OpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

token_provider = get_bearer_token_provider(
    DefaultAzureCredential(), "https://cognitiveservices.azure.com/.default"
)

client = OpenAI(  
  base_url = "https://finetuning-foundry.openai.azure.com/openai/v1/",  
  api_key=token_provider,
)

response = client.chat.completions.create(
  model="grok-4-fast-reasoning",  # Replace with your model deployment name.
    messages=[
        {"role": "system", "content": "Assistant is a large language model trained by OpenAI."},
        {"role": "user", "content": "Who were the founders of Microsoft?"}
    ]
)

#print(response)
print(response.model_dump_json(indent=2))
print(response.choices[0].message.content)

In [5]:
from datasets import load_from_disk

alpaca_train_formatted = load_from_disk("datasets/alpaca_train_formatted", keep_in_memory=True)

In [7]:
alpaca_test_formatted = load_from_disk("datasets/alpaca_test_formatted", keep_in_memory=True)

In [8]:
alpaca_train_formatted[0]

{'text': 'Beantwoord de volgende vraag zo goed mogelijk.\n\n### Vraag:\nSchrijf een gedicht van 7 regels met behulp van de gegeven woorden\ninput: paraplu, rail, water, lucht, gras, blad\n\n### Antwoord:\nOnder de donker wordende lucht,\nHet gras groeit nog steeds.\nEen eenzaam blad drijft weg\nVan de grijze ijzeren rail,\nDobberend als een paraplu in de golven van water.\nLangzaam dringt de nacht binnen, brengt rust en kalmte.\nEn ik sta en adem, voel de vrede in deze balsem.\n</s>'}

In [ ]:
### Side-by-Side Comparison of Baseline vs Finetuned Inference Results


import sys
import tempfile
from pathlib import Path

import pandas as pd

sys.path.insert(0, ".")
pd.set_option('display.max_colwidth', None)


from finetuning.config import load_config
from finetuning.blob_storage import download_blob_directory

config = load_config("configs/qlora_config.json")

# Derive names from config
run_name = config["wandb"]["run_name"]
storage_account = config["azureml"].get("storage_account", "llmaml5615532443")
inference_uri = config["azureml"]["inference_results_uri"]

# Parse container and blob prefix from the AzureML datastore URI
# Format: azureml://datastores/workspaceblobstore/paths/<blob_prefix>/
container = "azureml-blobstore-4c704101-7a51-4680-bcf8-f13966bf69b4"
blob_prefix = inference_uri.split("/paths/", 1)[1].strip("/")

# Download all inference results
tmp_dir = tempfile.mkdtemp()
download_blob_directory(storage_account, container, blob_prefix, tmp_dir)

# Load baseline and finetuned results using naming convention from run_inference.py
baseline_path = Path(tmp_dir) / "baseline_results.jsonl"
finetuned_path = Path(tmp_dir) / f"{run_name}_results.jsonl"

df_baseline = pd.read_json(baseline_path, lines=True)
df_finetuned = pd.read_json(finetuned_path, lines=True)

print(f"Baseline results:  {len(df_baseline)} rows")
print(f"Finetuned results: {len(df_finetuned)} rows ({run_name})")

Baseline results:  1500 rows
Finetuned results: 1500 rows (mistral_r16_a16_e1_b16_w30)


In [4]:
# Merge into a single side-by-side DataFrame
df_compare = df_baseline[["input", "expected_output"]].copy()
df_compare["baseline_output"] = df_baseline["predicted_output"]
df_compare["finetuned_output"] = df_finetuned["predicted_output"]

df_compare.head(3)

,input,expected_output,baseline_output,finetuned_output
0,Schrijf een korte alinea over het gegeven onderwerp.\ninput: Het belang van het gebruik van hernieuwbare energie,"Het gebruik van hernieuwbare energie groeit snel in relevantie en belang terwijl de wereld oplossingen zoekt om klimaatverandering tegen te gaan. Hernieuwbare energiebronnen, zoals zonne-, wind- en waterkracht, zijn duurzaam en hebben een veel lagere milieubelasting dan traditionele energiebronnen zoals kolen en olie. Overschakelen op hernieuwbare energie kan de uitstoot van koolstof verminderen, de luchtkwaliteit verbeteren, de afhankelijkheid van niet-hernieuwbare bronnen verminderen en zorgen voor een stabiele en veilige energietoekomst. Hernieuwbare energie is essentieel voor het bereiken van een duurzame toekomst en zou een belangrijk onderdeel moeten zijn van de energieportefeuille van elk land.","Het gebruik van hernieuwbare energie is van groot belang voor een duurzame toekomst. Het vermindert afhankelijkheid van fossiele brandstoffen, beperkt de uitstoot van broeikasgassen en bevordert de onafhankelijkheid van landen. Bovendien is hernieuwbare energie een steeds goedkopere en efficiëntere bron van energie.","Hernieuwbare energie is essentieel voor een duurzame toekomst. Het helpt om de uitstoot van broeikasgassen te verminderen en de afhankelijkheid van fossiele brandstoffen te verminderen. Hernieuwbare energiebronnen zoals zonne-energie, windenergie en waterkracht zijn schoon, duurzaam en kosteneffectief. Ze bieden een betrouwbare en duurzame energiebron voor huishoudens en bedrijven. Hernieuwbare energie is een belangrijk onderdeel van een duurzame toekomst en moet worden ondersteund door overheden en burgers over de hele wereld."
1,Maak een beschrijving voor een persoon met de volgende naam\ninput: John,"John is een lange, donkerharige man met een vriendelijke glimlach en een twinkeling in zijn ogen. Hij heeft een zachte, zorgzame aard en streeft er altijd naar om anderen te begrijpen. Hij heeft een passie voor leren en het verkennen van nieuwe plaatsen, ervaringen en culturen.","John is een man met de voornaam John. Hij is waarschijnlijk een volwassene, aangezien de meeste mensen met deze naam volwassenen zijn. Zijn geslacht is mannelijk. Over zijn uiterlijk, persoonlijkheid of andere kenmerken is geen informatie beschikbaar.",John is een vriendelijk en uitnodigend persoon. Hij is altijd bereid om te helpen en heeft een grote hart. Hij is een geweldige leraar en heeft een grote passie voor zijn werk. Hij is ook een geweldige vriend en heeft een grote liefde voor muziek en kunst.
2,Schrijf een verhaal dat begint met de volgende zin.\ninput: Ze opende de deur om een lange figuur gehuld in schaduwen te vinden.,"Met een trillende hand opende ze de deur, in de verwachting haar vriendin te vinden. In plaats daarvan stond er een lange figuur gehuld in schaduwen in de deuropening. Geschrokken door de onverwachte aanwezigheid, stapte ze achteruit en bonkte haar hart. De figuur maakte geen geluid en bleef stil staan, schijnbaar in gedachten verzonken. Ze wilde spreken, maar voordat ze een woord kon uitbrengen, stapte de figuur naar voren en onthulde een enkele roos in zijn hand. Terwijl ze de roos aannam, voelde ze een overweldigend gevoel van iets onbekends, iets wat ze niet kon uitleggen.","Ze opende de deur om een lange figuur gehuld in schaduwen te vinden. Het was een man met een hoog gezicht en een lang, grijsbaard. Hij droeg een oude, donkere jas en een wijdbrimmede hoed. Hij stond op het pad voor haar huis, met een zak in zijn hand.\n\nDe vrouw, die net haar deur had geopend, keek hem een ogenblik aan en vroeg: ""Kan ik je helpen?""\n\nDe man keek haar aan en antwoordde: ""Ik zoek een bepaald huis. Het is een oude, gele villa met een grote boom voor de deur.""\n\nDe vrouw herinnerde zich dat er een oude vriendin van haar in dat huis woonde. Ze vroeg: ""Zou je mij helpen als ik je daarheen begeleid?""\n\nDe man keek haar aan en antwoordde: ""Dat zou